# Fine-tune DistilBERT for Financial Sentiment

This notebook demonstrates tokenization, dataset preparation, training, evaluation, saving, and comparison between pretrained DistilBERT and a finance-tuned DistilBERT model.

In [ ]:
import numpy as np
import torch
from datasets import Dataset, load_dataset
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from transformers import AutoModelForSequenceClassification, AutoTokenizer, Trainer, TrainingArguments, pipeline

MODEL_NAME = "distilbert-base-uncased"
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

## Load Financial PhraseBank

Financial PhraseBank labels sentences as negative, neutral, or positive. If the dataset download is unavailable, the notebook uses a tiny built-in sample so every cell remains runnable.

In [ ]:
try:
    dataset = load_dataset("financial_phrasebank", "sentences_allagree", trust_remote_code=True)["train"]
    dataset = dataset.rename_column("sentence", "text")
except Exception as exc:
    print(f"Using fallback sample because dataset loading failed: {exc}")
    sample = {
        "text": [
            "The company reported strong revenue growth and higher profit.",
            "Shares fell after management lowered its full-year outlook.",
            "The firm maintained its previous guidance for the quarter.",
            "Demand improved and margins expanded significantly.",
            "The market reacted negatively to the earnings miss.",
            "Analysts described the update as broadly neutral."
        ],
        "label": [2, 0, 1, 2, 0, 1]
    }
    dataset = Dataset.from_dict(sample)

dataset = dataset.train_test_split(test_size=0.2, seed=SEED)
dataset

## Tokenization

A tokenizer converts human text into token IDs. DistilBERT uses subword tokens, so unknown financial words can be represented as smaller known pieces. Padding and truncation create fixed-size tensors for batching.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize(batch):
    return tokenizer(batch["text"], padding="max_length", truncation=True, max_length=128)

tokenized = dataset.map(tokenize, batched=True)
tokenized = tokenized.remove_columns(["text"])
tokenized.set_format("torch")
tokenized

## Model Setup

DistilBERT creates contextual embeddings with Transformer attention. Fine-tuning adds a classification head and updates weights so the model learns finance-specific sentiment language.

In [ ]:
id2label = {0: "negative", 1: "neutral", 2: "positive"}
label2id = {value: key for key, value in id2label.items()}
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=3,
    id2label=id2label,
    label2id=label2id
)

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, predictions, average="weighted", zero_division=0)
    accuracy = accuracy_score(labels, predictions)
    return {"accuracy": accuracy, "precision": precision, "recall": recall, "f1": f1}

In [ ]:
training_args = TrainingArguments(
    output_dir="../models/fine_tuned_distilbert",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=1,
    weight_decay=0.01,
    load_best_model_at_end=True,
    seed=SEED,
    report_to=[]
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["test"],
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)

trainer.train()

In [ ]:
fine_tuned_metrics = trainer.evaluate()
trainer.save_model("../models/fine_tuned_distilbert")
tokenizer.save_pretrained("../models/fine_tuned_distilbert")
fine_tuned_metrics

## Compare Pretrained vs Fine-tuned

The pretrained SST-2 checkpoint is binary and general-domain. The fine-tuned model is finance-domain and three-class. This simple comparison maps neutral examples to the closest non-neutral class only for demonstration; in a full research experiment, keep label spaces identical.

In [ ]:
sst2 = pipeline("sentiment-analysis", model="distilbert-base-uncased-finetuned-sst-2-english")
finance_model = pipeline("sentiment-analysis", model="../models/fine_tuned_distilbert", tokenizer="../models/fine_tuned_distilbert")

examples = [
    "The company beat earnings expectations and raised guidance.",
    "The stock dropped after weak demand and declining margins.",
    "Management repeated its earlier forecast."
]

for text in examples:
    print("TEXT:", text)
    print("Pretrained SST-2:", sst2(text)[0])
    print("Fine-tuned finance:", finance_model(text)[0])
    print()